# 🤖 Semantic Kernel Agent Framework

This notebook demonstrates how to build an **AI agent framework** using **Semantic Kernel** to handle multi-turn conversations with memory and workflow management.

**Documentation:**
- [Introduction to Semantic Kernel](https://learn.microsoft.com/en-us/semantic-kernel/overview/)
- [Semantic Kernel Agent Framework](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/?pivots=programming-language-python)

---

1. Install the `semantic-kernel` package
2. Create a ChatCompletion Service
3. Initialize Kernel and Add AI Service (OpenAI)
4. Create A ChatCompletion Agent 
5. Create a ChatCompletion Thread to maintain conversation history



## Installation & Setup



In [1]:
import importlib.util, subprocess, sys

_pkgs = ["semantic-kernel", "python-dotenv", "colorama"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

## Load Env Vars

### Setting up OpenAI secret Key  
1. Create an [OpenAI Account](https://platform.openai.com/signup/),
2. Go to [OpenAI's API Keys page](https://platform.openai.com/settings/organization/api-keys),
3. Click **Create new secret key** and copy it, 
4. You will need to add your billing information (MANAGE > Settings > Billing).

In [2]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

## Creating the ChatCompletion Service

In [3]:
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
import os

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4",
    api_key=os.getenv("OPENAI_API_KEY")
)

## Initialize the Kernel and add service

In [4]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.chat_completion_client_base import ChatCompletionClientBase

# Initialize the kernel
kernel = Kernel()

kernel.add_service(chat_completion_service, ChatCompletionClientBase)

## Creating a ChatCompletionAgent

In [5]:
from semantic_kernel.agents.chat_completion.chat_completion_agent import ChatCompletionAgent, ChatHistoryAgentThread

prompt = """
You are a Senior Developer Assistant. 
You think like a senior engineer with strong software architecture judgment and strong communication skills.
You help with:
  - coding and debugging
  - architecture decisions
  - refactoring & performance optimization
  - reviewing pull requests
  - troubleshooting production issues and fixing bugs
  - documenting systems
  - integrating APIs & services
  - DevOps & deployment guidance
  - rapid prototyping
You prioritize: clarity, maintainability, scalability, security, developer productivity, real-world best practices
"""
# Create the agent
agent = ChatCompletionAgent(
  kernel=kernel, 
  name="developer_agent", 
  instructions=prompt,
)

## Managing Conversation History

1. Define a **thread** to hold the conversation's context
2. If a **thread** is not created initially it will be created
3. and returned as part of the first response

**Documentation**: 
- [Maintain conversation history between invocations](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/chat-completion-agent?pivots=programming-language-python)
- [ChatHistoryAgentThread Class](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.chat_completion.chat_completion_agent.chathistoryagentthread?view=semantic-kernel-python)


In [6]:
thread = ChatHistoryAgentThread(thread_id="test_thread")

In [7]:
from colorama import Fore, Style

async def print_thread_summary(thread):
    print(f"{Fore.CYAN}Thread Summary:")
    if "thread" not in globals() or thread is None:
        print("No thread found. Run the chat cell first.")
    else:
        async def print_thread_messages(thread):
            async for msg in thread.get_messages():
                content = msg.content

                if isinstance(content, list):
                    content = " ".join(getattr(c, "content", str(c)) for c in content)
                elif not isinstance(content, str):
                    content = getattr(content, "content", str(content))

                print(f"{str(msg.role).upper()}: {content}\n {Style.RESET_ALL}")

        await print_thread_messages(thread)

### -- Mock Test to maintain conversation history --

In [8]:
# 📄 Mock Test: Thread Memory Operations

async def save_to_thread(agent, thread, messages):
    """Send messages and update thread correctly."""

    for msg in messages:
        print(f"{Fore.BLUE}USER → {msg}{Style.RESET_ALL}")

        # Single response call updates thread
        response = await agent.get_response(messages=msg, thread=thread)
        updated_thread = response.thread

        print(f"ASSISTANT → {response.content}\n")

    return updated_thread


async def count_messages(thread):
    """Count and retrieve messages from thread."""
    count = 0
    retrieved = []

    async for message in thread.get_messages():
        count += 1
        content = message.content

        if isinstance(content, list):
            content = " ".join(getattr(c, "content", str(c)) for c in content)
        elif not isinstance(content, str):
            content = getattr(content, "content", str(content))

        retrieved.append(f"{str(message.role).upper()}: {content}")

    return count, retrieved


async def test_thread_memory(agent, thread):
    """Test saving messages to the thread, counting them, and retrieving content."""

    test_messages = [
        "Write a Python function that converts a string into a URL-friendly slug",
        "Refactor previous Python function to improve its performance and readability",
        "Create a function that loads and validates a JSON configuration file",
        "Update loading and validation function with error handling and logging",
        "Write a function that reads a CSV file and returns a list of dictionaries"
    ]

    # Save test messages to thread
    updated_thread = await save_to_thread(agent, thread, test_messages)

    # Count messages in thread
    count, messages = await count_messages(updated_thread)
    print(f"\n✅ Total messages in thread: {count}")

    for m in messages:
        print(f"{Fore.CYAN}{m}{Style.RESET_ALL}")
        print("────────────────────────")

    # ✅ Assertions
    assert count >= len(test_messages), "Messages missing in thread!"
    assert any("json" in m.lower() for m in messages), "JSON task missing!"
    assert any("csv" in m.lower() for m in messages), "CSV task missing!"
    assert any("error handling" in m.lower() for m in messages), "Error handling task missing!"

    return updated_thread


# Run test
new_thread = await test_thread_memory(agent, thread)
thread = new_thread

# Inspect thread summary
print("CHAT HISTORY:")
async for msg in new_thread.get_messages():
    content = msg.content
    if isinstance(content, list):
        content = " ".join(getattr(c, "content", str(c)) for c in content)
    elif not isinstance(content, str):
        content = getattr(content, "content", str(content))

    print(f"{str(msg.role).upper()}: {content}\n{'-' * 30}")


USER → Write a Python function that converts a string into a URL-friendly slug
ASSISTANT → Sure, here is a Python function that converts a string to a URL-friendly slug using the built-in `re` and `unicodedata` packages:

```python
import re
import unicodedata

def slugify(value):
    """
    Convert string to a URL-friendly slug

    :param value: string 
    :return: string 
    """
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub('[^\w\s-]', '', value).strip().lower()
    return re.sub('[-\s]+', '-', value)
```

In this function:

- `unicodedata.normalize('NFKD', value)` will change any special characters like é to two separate characters 'e' and 'ˋ'. The 'NFKD' normalization splits special characters into their compatible decompositions.

- `.encode('ascii', 'ignore').decode('ascii')` will then convert the string to ASCII and ignore all characters that are not ASCII. This will remove 'ˋ'.

- `re.sub('[^\w\s-]', '', value)

## Conversing with Agent + Saving to Thread (short-memory)

In [9]:
from colorama import Fore, Style

async def run():
    global thread
    while True:
        print("======Thread Summary======")
        await print_thread_summary(thread)
        print("===========================")
        user_text = input("You: ").strip()
        if not user_text:
            print("Please enter a message.")
            continue
        if user_text.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break

        print(f"# User: {Fore.BLUE}{user_text}{Style.RESET_ALL}")
        print("Generating response...")
        response = await agent.get_response(messages=user_text, thread=thread)
        thread = response.thread

        print(f"\n{response.role}: {response.content}")

## Run the Agent

In [10]:
await run()

======Thread Summary======
Thread Summary:
AUTHORROLE.USER: Write a Python function that converts a string into a URL-friendly slug
 
AUTHORROLE.ASSISTANT: Sure, here is a Python function that converts a string to a URL-friendly slug using the built-in `re` and `unicodedata` packages:

```python
import re
import unicodedata

def slugify(value):
    """
    Convert string to a URL-friendly slug

    :param value: string 
    :return: string 
    """
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub('[^\w\s-]', '', value).strip().lower()
    return re.sub('[-\s]+', '-', value)
```

In this function:

- `unicodedata.normalize('NFKD', value)` will change any special characters like é to two separate characters 'e' and 'ˋ'. The 'NFKD' normalization splits special characters into their compatible decompositions.

- `.encode('ascii', 'ignore').decode('ascii')` will then convert the string to ASCII and ignore all characters that are n

You:  


Please enter a message.
======Thread Summary======
Thread Summary:
AUTHORROLE.USER: Write a Python function that converts a string into a URL-friendly slug
 
AUTHORROLE.ASSISTANT: Sure, here is a Python function that converts a string to a URL-friendly slug using the built-in `re` and `unicodedata` packages:

```python
import re
import unicodedata

def slugify(value):
    """
    Convert string to a URL-friendly slug

    :param value: string 
    :return: string 
    """
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub('[^\w\s-]', '', value).strip().lower()
    return re.sub('[-\s]+', '-', value)
```

In this function:

- `unicodedata.normalize('NFKD', value)` will change any special characters like é to two separate characters 'e' and 'ˋ'. The 'NFKD' normalization splits special characters into their compatible decompositions.

- `.encode('ascii', 'ignore').decode('ascii')` will then convert the string to ASCII and ignore a

You:  


Please enter a message.
======Thread Summary======
Thread Summary:
AUTHORROLE.USER: Write a Python function that converts a string into a URL-friendly slug
 
AUTHORROLE.ASSISTANT: Sure, here is a Python function that converts a string to a URL-friendly slug using the built-in `re` and `unicodedata` packages:

```python
import re
import unicodedata

def slugify(value):
    """
    Convert string to a URL-friendly slug

    :param value: string 
    :return: string 
    """
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub('[^\w\s-]', '', value).strip().lower()
    return re.sub('[-\s]+', '-', value)
```

In this function:

- `unicodedata.normalize('NFKD', value)` will change any special characters like é to two separate characters 'e' and 'ˋ'. The 'NFKD' normalization splits special characters into their compatible decompositions.

- `.encode('ascii', 'ignore').decode('ascii')` will then convert the string to ASCII and ignore a

You:  How do I write an async FastAPI endpoint?


# User: How do I write an async FastAPI endpoint?
Generating response...

AuthorRole.ASSISTANT: FastAPI is built to support both synchronous and asynchronous programming models. Here is how you might define a simple async endpoint using FastAPI:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/async-endpoint")
async def root():
    return {"message": "Hello, this is an async endpoint!"}
```

In this example:

- The `@app.get("/async-endpoint")` decorator tells FastAPI that the function below corresponds to an HTTP GET request at the `/async-endpoint` URL.

- The `async def root():` defines an asynchronous function. In Python, any function declared async will return an async object, which then needs to be awaited or run using an event loop.

- We return a simple JSON response immediately in this case, because this is a simple example that doesn't actually include asynchronous operations.

For a real-world asynchronous endpoint, you might fetch data from a database, ca

You:  quit


Goodbye!
